# 2. Importing FOF-CT imaging data

The [4DN FISH Omics Format — Chromatin Tracing](https://fish-omics-format.readthedocs.io/)
(FOF-CT) is the community standard for chromatin-tracing / multiplexed DNA
FISH experiments.  A FOF-CT *core table* is a CSV with a metadata header
(`##FOF-CT_Version=v1.0`, `##Columns=(…)`) followed by per-spot rows.

This notebook loads a real FOF-CT dataset with `ChromData.from_fofct` and
shows how to navigate the **Cell → Trace → Spot** hierarchy that imaging
data naturally exposes.


In [1]:
import numpy as np
import pandas as pd
from uchrom import ChromData

# Change this to a FOF-CT file you have locally
FOFCT = 'example-data/fofct_core.csv'  # or similar

# For this tutorial we build a synthetic example that looks like FOF-CT
# output in case you do not have a file handy.  Skip this cell if you
# downloaded a real file.
import io, os
if not os.path.exists(FOFCT):
    os.makedirs('example-data', exist_ok=True)
    rng = np.random.default_rng(0)
    header = '''##FOF-CT_Version=v1.0
##Table_Namespace=4dn_FOF-CT_core
##Genome_Assembly=GRCh38
##XYZ_Unit=micron
#Lab_Name: Demo
##Columns=(Spot_ID, Trace_ID, X, Y, Z, Chrom, Chrom_Start, Chrom_End, Cell_ID)
'''
    rows = []
    sid = 1
    for cell in range(20):
        for trace in range(2):          # 2 alleles per cell
            walk = np.cumsum(rng.normal(scale=0.12, size=(12, 3)), axis=0)
            for bin_i in range(12):      # 12 bins per trace
                x, y, z = walk[bin_i]
                start = 1_000_000 + bin_i * 10_000
                rows.append((sid, f'{cell}_{trace}', x, y, z,
                             'chr17', start, start + 10_000, cell))
                sid += 1
    with open(FOFCT, 'w') as f:
        f.write(header)
        for r in rows:
            f.write(','.join(str(x) for x in r) + '\n')

FOFCT


'example-data/fofct_core.csv'

## Load the file


In [2]:
cd = ChromData.read(FOFCT) if FOFCT.endswith(".h5cd") else ChromData.from_fofct(FOFCT)
cd


ChromData: n_spots=480, n_traces=40, n_cells=20
  spots:   ['chrom', 'start', 'end', 'trace_id', 'spot_id', 'cell_id']
  uns:     ['fofct_header', 'xyz_unit', 'genome_assembly']

## Inspect metadata

The FOF-CT header fields go into `cd.uns` so you can always trace the origin of a dataset.


In [3]:
cd.uns['fofct_header']


{'FOF-CT_Version': 'v1.0',
 'Table_Namespace': '4dn_FOF-CT_core',
 'Genome_Assembly': 'GRCh38',
 'XYZ_Unit': 'micron',
 'Lab_Name': 'Demo'}

In [4]:
print('genome   :', cd.uns.get('genome_assembly'))
print('xyz_unit :', cd.uns.get('xyz_unit'))
print()
print(cd.spots.head())


genome   : GRCh38
xyz_unit : micron

   chrom    start      end trace_id  spot_id cell_id
0  chr17  1000000  1010000      0_0        1       0
1  chr17  1010000  1020000      0_0        2       0
2  chr17  1020000  1030000      0_0        3       0
3  chr17  1030000  1040000      0_0        4       0
4  chr17  1040000  1050000      0_0        5       0


## Traces per chromosome

Imaging data typically has many traces per chromosome (each trace = one allele observation).


In [5]:
summary = (
    cd.spots
      .groupby('chrom', observed=True)
      .agg(n_spots=('trace_id', 'size'),
           n_traces=('trace_id', 'nunique'))
)
summary


,n_spots,n_traces
chrom,,
chr17,480,40


## Subsetting a single trace

This is the natural unit of analysis in chromatin tracing.


In [6]:
first_trace_id = cd.spots['trace_id'].iloc[0]
trace = cd.get_trace(first_trace_id)
print(trace)
print()
print(trace.to_dataframe().head())


ChromData: n_spots=12, n_traces=1, n_cells=1
  spots:   ['chrom', 'start', 'end', 'trace_id', 'spot_id', 'cell_id']
  uns:     ['fofct_header', 'xyz_unit', 'genome_assembly']

   chrom    start      end         x         y         z trace_id  spot_id  \
0  chr17  1000000  1010000  0.015088 -0.015853  0.076851      0_0        1   
1  chr17  1010000  1020000  0.027676 -0.080133  0.120242      0_0        2   
2  chr17  1020000  1030000  0.184156  0.033517  0.035794      0_0        3   
3  chr17  1030000  1040000  0.032305 -0.041276  0.040753      0_0        4   
4  chr17  1040000  1050000 -0.246699 -0.067531 -0.108756      0_0        5   

  cell_id  
0       0  
1       0  
2       0  
3       0  
4       0  


## Convert to ChromData-backed analysis format

Downstream analysis modules in `uchrom.fea` and `uchrom.strc` operate directly on the flat DataFrame produced by `cd.to_dataframe()` or accept the `ChromData` itself.


In [7]:
df = cd.to_dataframe()
print(df.columns.tolist())
df.head()


['chrom', 'start', 'end', 'x', 'y', 'z', 'trace_id', 'spot_id', 'cell_id']


,chrom,start,end,x,y,z,trace_id,spot_id,cell_id
0,chr17,1000000,1010000,0.015088,-0.015853,0.076851,0_0,1,0
1,chr17,1010000,1020000,0.027676,-0.080133,0.120242,0_0,2,0
2,chr17,1020000,1030000,0.184156,0.033517,0.035794,0_0,3,0
3,chr17,1030000,1040000,0.032305,-0.041276,0.040753,0_0,4,0
4,chr17,1040000,1050000,-0.246699,-0.067531,-0.108756,0_0,5,0


## Writing your own FOF-CT-compatible data

If you produced tracing data with a tool that does not write FOF-CT, you
can create a ChromData directly (see Tutorial 1) and write `.h5cd`.  The
on-disk format is versioned so downstream tools can safely consume it.
